<a href="https://colab.research.google.com/github/kylashrao/DataScience-Projects/blob/main/Advanced_AML_Fraud_Detection_using_Isolation_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

End-to-End Project: Advanced AML / Fraud Detection using Isolation Forest
We will build an advanced unsupervised machine learning pipeline to detect money laundering or anomalous transaction behavior using Pandas, NumPy, Scikit-learn, and SciPy. Isolation Forest isolates anomalies instead of profiling normal data points, making it highly effective for rare financial crimes.

1. Import Libraries & Generate Complex Financial Dataset

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import zscore
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Set random seed
np.random.seed(42)

# Simulating advanced transaction data (Normal transactions + Hidden Anomalies)
n_samples = 1000
transaction_amount = np.random.exponential(scale=200, size=n_samples)
transaction_frequency = np.random.poisson(lam=3, size=n_samples)
account_age_months = np.random.randint(1, 120, size=n_samples)

# Inject synthetic anomalies (e.g., sudden massive spikes in frequency and amount)
anomaly_indices = np.random.choice(n_samples, size=50, replace=False)
transaction_amount[anomaly_indices] = np.random.uniform(
    5000, 20000, size=50
)
transaction_frequency[anomaly_indices] = np.random.randint(25, 50, size=50)

# Ground truth for evaluation (1 for normal, -1 for anomaly in Isolation Forest terms, mapped to 1 for fraud)
true_labels = np.zeros(n_samples)
true_labels[anomaly_indices] = 1

df = pd.DataFrame(
    {
        'Amount': transaction_amount,
        'Frequency': transaction_frequency,
        'Account_Age': account_age_months,
    }
)

2. Advanced Feature Engineering & Statistical Scaling
Financial models benefit from robust feature transformations. We will compute rolling or statistical ratios using SciPy/Pandas and scale features.

In [3]:
# Create advanced interaction features
df['Amount_Per_Frequency'] = df['Amount'] / (df['Frequency'] + 1)
df['Z_Score_Amount'] = np.abs(zscore(df['Amount']))

# Define feature matrix X
X = df[['Amount', 'Frequency', 'Account_Age', 'Amount_Per_Frequency', 'Z_Score_Amount']]

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

3. Train the Isolation Forest Model
Isolation Forest isolates observations by randomly selecting a feature and a split value. Anomalies require fewer splits to isolate than normal data points.

In [4]:
# Initialize Isolation Forest (contamination is the expected proportion of outliers)
iso_forest = IsolationForest(
    contamination=0.05, random_state=42, n_estimators=200
)

# Fit the model and predict (-1 for anomalies/fraud, 1 for normal)
predictions = iso_forest.fit_predict(X_scaled)

# Convert predictions to binary: -1 (anomaly) -> 1 (fraud), 1 (normal) -> 0
pred_labels = np.where(predictions == -1, 1, 0)

4. Model Evaluation
Evaluate how effectively the unsupervised model caught the injected synthetic fraud rings without prior training labels.

In [5]:
# Evaluation Metrics
print('--- Advanced AML Anomaly Detection Performance ---')
print('Confusion Matrix:')
print(confusion_matrix(true_labels, pred_labels))
print('\nClassification Report:')
print(classification_report(true_labels, pred_labels))

--- Advanced AML Anomaly Detection Performance ---
Confusion Matrix:
[[949   1]
 [  1  49]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       950
         1.0       0.98      0.98      0.98        50

    accuracy                           1.00      1000
   macro avg       0.99      0.99      0.99      1000
weighted avg       1.00      1.00      1.00      1000

